# System-level metrics across costs

Script for AAL. To use a different atlas, simply change the paths to import the desired parcellation.

fMRI graphs

In [ ]:
import numpy as np
import networkx as nx
import scipy.special as ss
from networkx import tree
import os
import glob

def adj_matrix_connected(corr_matrix,sparsity_value):
    """given the correlation matrix and the expected sparsity coefficient it can 
    happen that the corresponding thresholded matrix results in a disconnected graph
    here we force the graph to be fully connected by the computation of the minimum
    spanning tree and adding the required edges in order to have a unique connected component 
    """
    if sparsity_value == 1.0:
        adj_matrix=np.ones(corr_matrix.shape)
        np.fill_diagonal(adj_matrix,0)
        return adj_matrix
        
    
    corr_matrix =abs(corr_matrix)

    max_num_edges = ss.comb(corr_matrix.shape[0],2)
    num_edges = int(max_num_edges*sparsity_value)
    
    num_regions=corr_matrix.shape[0]
    #total number of regions in the graph
        
    totalgraph=nx.from_numpy_array(1-abs(corr_matrix))
    #extraction of a complete graph having has weight 1-abs(correlation)
    #we need to take 1-abs since the mst is taking the minimum weight graph and we want the most correlated edges to be there
    
    MST=nx.to_numpy_array(tree.minimum_spanning_tree(totalgraph).to_undirected())
    MST_adj_mat=MST
    MST_adj_mat[MST>0]==1
    MST_adj_mat=np.triu(MST_adj_mat) #put zeros in the inferior triangular matrix
    
    #put zeros in the diagonal of the corr matrix
    for i in range(num_regions):
        corr_matrix[i,i]=0
    
    values_corr=abs(np.triu(corr_matrix))
    
    cor_wo_MST=values_corr[np.triu(MST_adj_mat)==0]
    #we do not consider the correlation values which do not involve edges that are already in the MST
    
    values=list(cor_wo_MST.flatten())
    values.sort(reverse=True)
    
    #we select the maximum value of correlation to have the expected num of edges - num of edges in the mst (num regions-1)
    value_thresh=values[num_edges-(num_regions-1)-1] #-1 index start at 0
    
    adj_matrix=np.zeros(corr_matrix.shape) 
    
    #we put an edge if the value of correlation is higher than the found threshold or if the edges is required by the mst
    adj_matrix[values_corr>=value_thresh]=1
    adj_matrix[MST_adj_mat!=0]=1
    
    adj_matrix=np.triu(adj_matrix)+np.transpose(np.triu(adj_matrix)) #simmetry of the adj matrix
    
    return adj_matrix


In [ ]:
controls_aal = [
    "01FO", "02LE", "03GA", "04GM", "05IM", "07NA", "08CP", "09DM", "11GL", "12LJ",
    "13AE", "14PM", "15GT", "16DT", "17LY", "19DG", "20CP", "21LJ", "22DD", "23BA"
]

anoxic_aal = [ # Effective Anoxic patients
    "01JF", "02PD", "06BM", "07TA", "14RC"
]
traumatic_aal = [ # Effective Traumatic patients   - Patient 08PE must be excluded from studies using AICHA 
    "03DB", "08PE", "11FC", "13TL", "16FF", "22BT", "23GC", "24ZX", "26AC"
]

matrices_path = "/Data/Correlation_Matrices/AAL" # Simply change the last path to import other atlases, e.g. AICHA 
control_path = os.path.join(matrices_path, "Controls")
anoxic_path = os.path.join(matrices_path, "Anoxic")
traumatic_path = os.path.join(matrices_path, "Traumatic")

# Final filter before importing - matching IDs from lists to files to exclude non-effective subjects

control_subjects = sorted(
    s for s in glob.glob(os.path.join(control_path, "*"))
    if os.path.basename(s).split(".")[0] in controls_aal
)

anoxic_subjects = sorted(
    s for s in glob.glob(os.path.join(anoxic_path, "*"))
    if os.path.basename(s).split(".")[0] in anoxic_aal
)

traumatic_subjects = sorted(
    s for s in glob.glob(os.path.join(traumatic_path, "*"))
    if os.path.basename(s).split(".")[0] in traumatic_aal
)

# Importing correlations - Keys are IDs, values are matrices

control_correlations = {}
for sub in control_subjects:
    control_correlations[os.path.splitext(os.path.basename(sub))[0]] = np.loadtxt(sub)


traumatic_correlations = {}
for sub in traumatic_subjects:
    traumatic_correlations[os.path.splitext(os.path.basename(sub))[0]] = np.loadtxt(sub)

anoxic_correlations = {}
for sub in anoxic_subjects:
    anoxic_correlations[os.path.splitext(os.path.basename(sub))[0]] = np.loadtxt(sub)

print(f"Controls: {len(control_correlations.keys())}, Anoxic: {len(anoxic_correlations.keys())}, Traumatic: {len(traumatic_correlations.keys())}")

# Constructing fMRI graphs
costs = np.linspace(0.05, 0.5, 10)
controls_fmri_graphs = {cost:{sub: None for sub in control_correlations.keys()} for cost in costs}
for cost in costs:
    for sub in control_correlations.keys(): # Keys are IDs, values are nx.Graph objects
        controls_fmri_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(control_correlations[sub], cost))

anoxic_fmri_graphs = {cost:{sub: None for sub in anoxic_correlations.keys()} for cost in costs}
for cost in costs:
    for sub in anoxic_correlations.keys():
        anoxic_fmri_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(anoxic_correlations[sub], cost))

traumatic_fmri_graphs = {cost:{sub: None for sub in traumatic_correlations.keys()} for cost in costs}
for cost in costs:
    for sub in traumatic_correlations.keys():
        traumatic_fmri_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(traumatic_correlations[sub], cost))

# We can merge the coma dataset in one dictionary

coma_fmri_graphs = {cost: {} for cost in costs}

for cost in costs:
    coma_fmri_graphs[cost] = {**anoxic_fmri_graphs[cost], **traumatic_fmri_graphs[cost]}


Controls: 20, Anoxic: 5, Traumatic: 9


In [3]:
favorable_group = ["06BM", "11FC", "13TL", "14RC", "22BT", "23GC", "24ZX", "25AY", "26AC"]
unfavorable_group = ["01JF", "02PD", "03DB", "07TA", "08PE", "09CD", "16FF", "20MP"]

PET data

In [ ]:
tspo_matrix_path = "/Data/PET_Matrices/AAL"

########### Controls
list_tspo_cn = os.listdir(os.path.join(tspo_matrix_path, "Controls"))

controls_tspo_matrices = {}

for f in list_tspo_cn:
    #sub = 
    sub = f.split(".")[0]
    controls_tspo_matrices[sub] = np.loadtxt(os.path.join(tspo_matrix_path, "Controls", f), delimiter=",")

list_tspo_anox = os.listdir(os.path.join(tspo_matrix_path, "Anoxic"))


########### Anoxic
# --- Initialize container ---
anoxic_tspo_matrices = {}

# --- Load matrices, excluding unwanted IDs ---
for f in list_tspo_anox:
    sub = f.split(".")[0]  # extract subject ID (before .csv or .txt)
    
    if sub not in anoxic_aal:
        print(f"Skipping excluded subject: {sub}")
        continue  # skip loading this subject
    
    file_path = os.path.join(tspo_matrix_path, "Anoxic", f)
    anoxic_tspo_matrices[sub] = np.loadtxt(file_path, delimiter=",")


########### Traumatic

list_tspo_trau = os.listdir(os.path.join(tspo_matrix_path, "Traumatic"))

# --- Initialize container ---
traumatic_tspo_matrices = {}

# --- Load matrices, excluding unwanted IDs ---
for f in list_tspo_trau:
    sub = f.split(".")[0]  # extract subject ID (before .csv or .txt)
    
    if sub not in traumatic_aal:
        print(f"Skipping excluded subject: {sub}")
        continue  # skip loading this subject
    
    file_path = os.path.join(tspo_matrix_path, "Traumatic", f)
    traumatic_tspo_matrices[sub] = np.loadtxt(file_path, delimiter=",")

controls_tspo_graphs = {cost:{sub: None for sub in controls_tspo_matrices.keys()} for cost in costs}
for cost in costs:
    for sub in controls_tspo_matrices.keys():
        controls_tspo_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(controls_tspo_matrices[sub], cost))

anoxic_tspo_graphs = {cost:{sub: None for sub in anoxic_tspo_matrices.keys()} for cost in costs}
for cost in costs:
    for sub in anoxic_tspo_matrices.keys():
            anoxic_tspo_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(anoxic_tspo_matrices[sub], cost))

traumatic_tspo_graphs = {cost:{sub: None for sub in traumatic_tspo_matrices.keys()} for cost in costs}
for cost in costs:
    for sub in traumatic_tspo_matrices.keys():
        traumatic_tspo_graphs[cost][sub] = nx.from_numpy_array(adj_matrix_connected(traumatic_tspo_matrices[sub], cost))

# We can merge the coma dataset in one dictionary

coma_tspo_graphs = {cost: {} for cost in costs}

for cost in costs:
    coma_tspo_graphs[cost] = {**anoxic_tspo_graphs[cost], **traumatic_tspo_graphs[cost]}


Skipping excluded subject: 09CD
Skipping excluded subject: 20MP
Skipping excluded subject: 25AY


Graph functions: global metrics

In [ ]:

def global_graph_metrics(G):
    metrics = {}

    # Global efficiency
    metrics['global_efficiency'] = nx.global_efficiency(G)
    # Average clustering
    metrics['avg_clustering'] = nx.average_clustering(G)
    # Average path length
    metrics['avg_path_length'] = nx.average_shortest_path_length(G)
    # Small-worldness: using formula S = (C_real / C_rand) / (L_real / L_rand)
    n = G.number_of_nodes()
    p = nx.density(G)
    rand_graph = nx.erdos_renyi_graph(n, p, seed=246) # All analyses use this seed

    if nx.is_connected(rand_graph):
        L_rand = nx.average_shortest_path_length(rand_graph)
    else:
        largest_cc_rand = max(nx.connected_components(rand_graph), key=len)
        subgraph_rand = rand_graph.subgraph(largest_cc_rand)
        L_rand = nx.average_shortest_path_length(subgraph_rand)

    C_rand = nx.average_clustering(rand_graph)
    S = (metrics['avg_clustering'] / C_rand) / (metrics['avg_path_length'] / L_rand)
    metrics['small_worldness'] = S


    return metrics

fMRI and PET metrics at all costs

In [6]:
coma_aal = anoxic_aal + traumatic_aal

controls_tspo_graphs_metrics = {cost:{sub: None for sub in controls_aal} for cost in costs} # considering graphs at 5 and 10%
for cost in costs:
    for sub in controls_aal:
        controls_tspo_graphs_metrics[cost][sub] = global_graph_metrics(controls_tspo_graphs[cost][sub])

coma_tspo_graphs_metrics = {cost:{sub: None for sub in coma_aal} for cost in costs} # considering graphs at 5 and 10%
for cost in costs:
    for sub in coma_aal:
        coma_tspo_graphs_metrics[cost][sub] = global_graph_metrics(coma_tspo_graphs[cost][sub])

controls_fmri_graphs_metrics = {cost:{sub: None for sub in controls_aal} for cost in costs} # considering graphs at 5 and 10%
for cost in costs:
    for sub in controls_aal:
        controls_fmri_graphs_metrics[cost][sub] = global_graph_metrics(controls_fmri_graphs[cost][sub])

coma_fmri_graphs_metrics = {cost:{sub: None for sub in coma_aal} for cost in costs} # considering graphs at 5 and 10%
for cost in costs:
    for sub in coma_aal:
        coma_fmri_graphs_metrics[cost][sub] = global_graph_metrics(coma_fmri_graphs[cost][sub])

U-tests for global graph metrics following the usual order: Coma (full cohort), Etiologies (AE, TE) and Outcomes (FO, UO)

In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu


def metrics_to_df(metrics_dict, group_name, cost):
    records = []
    for sub, metrics in metrics_dict[cost].items():
        for metric_name, value in metrics.items():
            records.append({"subject": sub, "group": group_name, "cost": cost,
                            "metric": metric_name, "value": value})
    return pd.DataFrame(records)

def run_mannwhitney_tests(metrics_df, group_col="group", group_pairs=None):
    results = []
    for cost in metrics_df["cost"].unique():
        for metric in metrics_df["metric"].unique():
            if group_pairs is None:
                unique_groups = metrics_df[group_col].unique()
                group_pairs = [(g1, g2) for i, g1 in enumerate(unique_groups)
                               for g2 in unique_groups[i+1:]]
            for g1, g2 in group_pairs:
                vals1 = metrics_df[(metrics_df[group_col] == g1) & 
                                   (metrics_df["cost"] == cost) & 
                                   (metrics_df["metric"] == metric)]["value"]
                vals2 = metrics_df[(metrics_df[group_col] == g2) & 
                                   (metrics_df["cost"] == cost) & 
                                   (metrics_df["metric"] == metric)]["value"]

                n1 = len(vals1)
                n2 = len(vals2)

                if n1 > 0 and n2 > 0:
                    stat, pval = mannwhitneyu(vals1, vals2, alternative="two-sided")
                    results.append({
                        "cost": cost,
                        "metric": metric,
                        "group1": g1,
                        "group2": g2,
                        "n1": n1,   
                        "n2": n2,   
                        "U": stat,
                        "p": pval
                    })
    return pd.DataFrame(results)




In [8]:
# -----------------------------
# Controls vs Coma
# -----------------------------

# TSPO

all_dfs = []
for cost in costs:
    df_controls = metrics_to_df(controls_tspo_graphs_metrics, "Controls", cost)
    df_coma = metrics_to_df(coma_tspo_graphs_metrics, "Coma", cost)
    all_dfs.append(pd.concat([df_controls, df_coma], ignore_index=True))

metrics_df_tspo = pd.concat(all_dfs, ignore_index=True)

results_df_tspo = run_mannwhitney_tests(metrics_df_tspo, group_pairs=[("Controls", "Coma")])
print("PET")
print(results_df_tspo)


# fMRI

all_dfs = []
for cost in costs:
    df_controls = metrics_to_df(controls_fmri_graphs_metrics, "Controls", cost)
    df_coma = metrics_to_df(coma_fmri_graphs_metrics, "Coma", cost)
    all_dfs.append(pd.concat([df_controls, df_coma], ignore_index=True))

metrics_df_fmri = pd.concat(all_dfs, ignore_index=True)

results_df_fmri = run_mannwhitney_tests(metrics_df_fmri, group_pairs=[("Controls", "Coma")])
print("\nfMRI")
print(results_df_fmri)


# -----------------------------
# Controls vs Etiologies
# -----------------------------

# TSPO/PET
all_dfs = []

for cost in costs:
    df_controls = metrics_to_df(controls_tspo_graphs_metrics, "Controls", cost)
    
    # Filter coma_tspo_graphs_metrics for anoxic and traumatic subjects
    anoxic_metrics = {
        cost: {sub: coma_tspo_graphs_metrics[cost][sub] 
               for sub in coma_tspo_graphs_metrics[cost] if sub in anoxic_aal}
    }
    traumatic_metrics = {
        cost: {sub: coma_tspo_graphs_metrics[cost][sub] 
               for sub in coma_tspo_graphs_metrics[cost] if sub in traumatic_aal}
    }
    
    # Only create dataframes if there are subjects
    df_anoxic = metrics_to_df(anoxic_metrics, "AE", cost) if anoxic_metrics[cost] else pd.DataFrame()
    df_traumatic = metrics_to_df(traumatic_metrics, "TE", cost) if traumatic_metrics[cost] else pd.DataFrame()
    
    all_dfs.append(pd.concat([df_controls, df_anoxic, df_traumatic], ignore_index=True))

metrics_df_tspo_etiologies = pd.concat(all_dfs, ignore_index=True)

results_df_tspo_eti = run_mannwhitney_tests(metrics_df_tspo_etiologies)
print("Controls vs Anoxic vs Traumatic (pairwise tests) PET:")
print(results_df_tspo_eti)

#fMRI

all_dfs = []

for cost in costs:
    df_controls = metrics_to_df(controls_fmri_graphs_metrics, "Controls", cost)
    
    # Filter coma_fmri_graphs_metrics for anoxic and traumatic subjects
    anoxic_metrics = {
        cost: {sub: coma_fmri_graphs_metrics[cost][sub] 
               for sub in coma_fmri_graphs_metrics[cost] if sub in anoxic_aal}
    }
    traumatic_metrics = {
        cost: {sub: coma_fmri_graphs_metrics[cost][sub] 
               for sub in coma_fmri_graphs_metrics[cost] if sub in traumatic_aal}
    }
    
    # Only create dataframes if there are subjects
    df_anoxic = metrics_to_df(anoxic_metrics, "AE", cost) if anoxic_metrics[cost] else pd.DataFrame()
    df_traumatic = metrics_to_df(traumatic_metrics, "TE", cost) if traumatic_metrics[cost] else pd.DataFrame()
    
    all_dfs.append(pd.concat([df_controls, df_anoxic, df_traumatic], ignore_index=True))

metrics_df_fmri_etiologies = pd.concat(all_dfs, ignore_index=True)

results_df_fmri_eti = run_mannwhitney_tests(metrics_df_fmri_etiologies)
print("Controls vs Anoxic vs Traumatic (pairwise tests) fMRI:")
print(results_df_fmri_eti)

# -----------------------------
# Controls vs Outcomes
# -----------------------------

# PET

all_dfs = []

for cost in costs:
    df_controls = metrics_to_df(controls_tspo_graphs_metrics, "Controls", cost)
    
    # Filter coma_tspo_graphs_metrics for anoxic and traumatic subjects
    fo_metrics = {
        cost: {sub: coma_tspo_graphs_metrics[cost][sub] 
               for sub in coma_tspo_graphs_metrics[cost] if sub in favorable_group}
    }
    uo_metrics = {
        cost: {sub: coma_tspo_graphs_metrics[cost][sub] 
               for sub in coma_tspo_graphs_metrics[cost] if sub in unfavorable_group}
    }
    
    # Only create dataframes if there are subjects
    df_fo = metrics_to_df(fo_metrics, "FO", cost) if fo_metrics[cost] else pd.DataFrame()
    df_uo = metrics_to_df(uo_metrics, "UO", cost) if uo_metrics[cost] else pd.DataFrame()
    
    all_dfs.append(pd.concat([df_controls, df_fo, df_uo], ignore_index=True))

metrics_df_tspo_outcome = pd.concat(all_dfs, ignore_index=True)

results_df_tspo_outcome = run_mannwhitney_tests(metrics_df_tspo_outcome)
print("Controls vs FO vs UO (pairwise tests) PET:")
print(results_df_tspo_outcome)


# fMRI

all_dfs = []

for cost in costs:
    df_controls = metrics_to_df(controls_fmri_graphs_metrics, "Controls", cost)
    
    # Filter coma_fmri_graphs_metrics for anoxic and traumatic subjects
    fo_metrics = {
        cost: {sub: coma_fmri_graphs_metrics[cost][sub] 
               for sub in coma_fmri_graphs_metrics[cost] if sub in favorable_group}
    }
    uo_metrics = {
        cost: {sub: coma_fmri_graphs_metrics[cost][sub] 
               for sub in coma_fmri_graphs_metrics[cost] if sub in unfavorable_group}
    }
    
    # Only create dataframes if there are subjects
    df_fo = metrics_to_df(fo_metrics, "FO", cost) if fo_metrics[cost] else pd.DataFrame()
    df_uo = metrics_to_df(uo_metrics, "UO", cost) if uo_metrics[cost] else pd.DataFrame()
    
    all_dfs.append(pd.concat([df_controls, df_fo, df_uo], ignore_index=True))

metrics_df_fmri_outcome = pd.concat(all_dfs, ignore_index=True)

results_df_fmri_outcome = run_mannwhitney_tests(metrics_df_fmri_outcome)
print("\nControls vs FO vs UO (pairwise tests) fMRI:")
print(results_df_fmri_eti)


PET
    cost             metric    group1 group2  n1  n2      U         p
0   0.05  global_efficiency  Controls   Coma  20  14  103.0  0.201520
1   0.05     avg_clustering  Controls   Coma  20  14   96.0  0.127963
2   0.05    avg_path_length  Controls   Coma  20  14  174.0  0.241094
3   0.05    small_worldness  Controls   Coma  20  14   83.0  0.048032
4   0.10  global_efficiency  Controls   Coma  20  14   78.0  0.031393
5   0.10     avg_clustering  Controls   Coma  20  14  116.0  0.410890
6   0.10    avg_path_length  Controls   Coma  20  14  192.0  0.071526
7   0.10    small_worldness  Controls   Coma  20  14   97.0  0.136965
8   0.15  global_efficiency  Controls   Coma  20  14   61.0  0.006016
9   0.15     avg_clustering  Controls   Coma  20  14  110.0  0.301939
10  0.15    avg_path_length  Controls   Coma  20  14  213.0  0.011182
11  0.15    small_worldness  Controls   Coma  20  14   77.0  0.028739
12  0.20  global_efficiency  Controls   Coma  20  14   63.0  0.007430
13  0.20     avg

Export

In [ ]:
# -----------------------------
# EXPORT RESULTS TO CSV
# -----------------------------

# Controls vs Coma
import os
exp_pth = "/C"
results_df_tspo.to_csv(os.path.join(exp_pth,"AAL89_coma_pet.csv"), index=False)
results_df_fmri.to_csv(os.path.join(exp_pth,"AAL89_coma_fmri.csv"), index=False)

# Controls vs Etiologies
results_df_tspo_eti.to_csv(os.path.join(exp_pth,"AAL89_etiologies_pet.csv"), index=False)
results_df_fmri_eti.to_csv(os.path.join(exp_pth,"AAL89_etiologies_fmri.csv"), index=False)

# Controls vs Outcomes
results_df_tspo_outcome.to_csv(os.path.join(exp_pth,"AAL89_outcomes_pet.csv"), index=False)
results_df_fmri_outcome.to_csv(os.path.join(exp_pth,"AAL89_outcomes_fmri.csv"), index=False)